# Campaign Simulator

Turn our model into a decision tool and answer what-if business questions.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from src.decision_engine.campaign_optimizer import CampaignConfig, rank_customers

# Re-run necessary setup steps
df = pd.read_csv("../data/processed/model_data.csv")
X = df.drop(columns=["y"])
y = df["y"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numerical_features),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)
])

xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42))
])
xgb_model.fit(X_train, y_train)

all_probability = xgb_model.predict_proba(X)[:, 1]
decision_df = X.copy()
decision_df["xgb_probability"] = all_probability
decision_df["actual"] = y.values

config = CampaignConfig(conversion_value=1000, contact_cost=20, budget=100000)
ranked_customers = rank_customers(decision_df, probability_column="xgb_probability", config=config)


In [ ]:
from src.decision_engine.simulator import (
    CampaignScenario,
    simulate_campaign
)

scenario = CampaignScenario(
    budget=100000,
    contact_cost=20,
    conversion_value=1000
)

result = simulate_campaign(
    ranked_customers,
    scenario
)


In [ ]:
print(f"Customers targeted: {result['customers_targeted']:,}")
print(f"Expected conversions: {result['expected_conversions']:.2f}")
print(f"Campaign cost: ₹{result['campaign_cost']:,.2f}")
print(f"Expected revenue: ₹{result['expected_revenue']:,.2f}")
print(f"Expected profit: ₹{result['expected_profit']:,.2f}")
print(f"Expected ROI: {result['expected_roi']:.2f}x")

In [ ]:
budgets = [25000, 50000, 100000, 200000, 500000]

scenario_results = []

for budget in budgets:
    scenario = CampaignScenario(
        budget=budget,
        contact_cost=20,
        conversion_value=1000
    )

    result = simulate_campaign(
        ranked_customers,
        scenario
    )

    scenario_results.append({
        "budget": budget,
        "customers_targeted": result["customers_targeted"],
        "expected_conversions": result["expected_conversions"],
        "expected_revenue": result["expected_revenue"],
        "expected_profit": result["expected_profit"],
        "expected_roi": result["expected_roi"]
    })

scenario_df = pd.DataFrame(scenario_results)
scenario_df

In [ ]:
contact_costs = [10, 20, 40, 60, 100]

cost_results = []

for cost in contact_costs:
    scenario = CampaignScenario(
        budget=100000,
        contact_cost=cost,
        conversion_value=1000
    )

    result = simulate_campaign(
        ranked_customers,
        scenario
    )

    cost_results.append({
        "contact_cost": cost,
        "customers_targeted": result["customers_targeted"],
        "expected_conversions": result["expected_conversions"],
        "expected_profit": result["expected_profit"],
        "expected_roi": result["expected_roi"]
    })

cost_df = pd.DataFrame(cost_results)
cost_df

In [ ]:
conversion_values = [500, 1000, 2000, 5000]

value_results = []

for value in conversion_values:
    scenario = CampaignScenario(
        budget=100000,
        contact_cost=20,
        conversion_value=value
    )

    result = simulate_campaign(
        ranked_customers,
        scenario
    )

    value_results.append({
        "conversion_value": value,
        "customers_targeted": result["customers_targeted"],
        "expected_conversions": result["expected_conversions"],
        "expected_profit": result["expected_profit"],
        "expected_roi": result["expected_roi"]
    })

value_df = pd.DataFrame(value_results)
value_df

In [ ]:
def random_campaign(
    df,
    number_of_customers,
    conversion_value,
    contact_cost,
    random_state=42
):
    sample = df.sample(
        n=min(number_of_customers, len(df)),
        random_state=random_state
    )

    expected_conversions = sample["actual"].mean() * len(sample)
    cost = len(sample) * contact_cost
    revenue = expected_conversions * conversion_value
    profit = revenue - cost
    roi = profit / cost if cost > 0 else 0

    return {
        "customers": len(sample),
        "expected_conversions": expected_conversions,
        "cost": cost,
        "revenue": revenue,
        "profit": profit,
        "roi": roi
    }

# Test random baseline
random_res = random_campaign(decision_df, result['customers_targeted'], 1000, 20)
print("Random Campaign Profit: ₹{:,.2f}".format(random_res['profit']))
print("Targeted Campaign Profit: ₹{:,.2f}".format(result['expected_profit']))